In [2]:
import numpy as np
from collections import OrderedDict

In [3]:
x = np.random.randn(10, 1, 28, 28)

## im2col을 활용해서 데이터 전개하기
+ https://compmath.korea.ac.kr/deeplearning/ConvolutionNN.html#im2col
  
  

In [4]:
def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    """다수의 이미지를 입력받아 2차원 배열로 변환한다(평탄화).
    
    Parameters
    ----------
    input_data : 4차원 배열 형태의 입력 데이터(이미지 수, 채널 수, 높이, 너비)
    filter_h : 필터의 높이
    filter_w : 필터의 너비
    stride : 스트라이드
    pad : 패딩
    
    Returns
    -------
    col : 2차원 배열
    """
    N, C, H, W = input_data.shape
    out_h = (H + 2*pad - filter_h)//stride + 1
    out_w = (W + 2*pad - filter_w)//stride + 1

    img = np.pad(input_data, [(0,0), (0,0), (pad, pad), (pad, pad)], 'constant')
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w))

    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]
            print(col, sep='\n\n')

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)
    ## N, OH, OW, C, FH, FW로 전치한 뒤, N*OH*OW, C*FH*FW 크기의 배열로 변경한다.
    return col

def col2im(col, input_shape, filter_h, filter_w, stride=1, pad=0):
    N, C, H, W = input_shape
    out_h = (H + 2*pad - filter_h)//stride + 1
    out_w = (W + 2*pad - filter_w)//stride + 1
    col = col.reshape(N, out_h, out_w, C, filter_h, filter_w).transpose(0, 3, 4, 5, 1, 2)

    img = np.zeros((N, C, H + 2*pad + stride - 1, W + 2*pad + stride - 1))
    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            img[:, :, y:y_max:stride, x:x_max:stride] += col[:, :, y, x, :, :]

    return img[:, :, pad:H + pad, pad:W + pad]

In [5]:
m = np.arange(5)
n = np.random.randn(5,5)

n[0] = m
print(n)

[[ 0.          1.          2.          3.          4.        ]
 [-1.15678491  0.19157033  0.71422595 -0.3866349  -0.48326155]
 [-0.94011531 -1.18590754 -0.94406334  0.34925127 -0.75840667]
 [-1.3033688  -0.86775365  0.10909318  0.88401819 -0.47527645]
 [ 0.17059427 -1.10922888  0.45263627 -0.02156651  0.47898211]]


In [6]:
x1 = np.arange(1*3*3*3).reshape(1,3,3,3)
print(x1)
col1 = im2col(x1, 2, 2, stride=1, pad=0)
print(col1, col1.shape)

[[[[ 0  1  2]
   [ 3  4  5]
   [ 6  7  8]]

  [[ 9 10 11]
   [12 13 14]
   [15 16 17]]

  [[18 19 20]
   [21 22 23]
   [24 25 26]]]]
[[[[[[ 0.  1.]
     [ 3.  4.]]

    [[ 0.  0.]
     [ 0.  0.]]]


   [[[ 0.  0.]
     [ 0.  0.]]

    [[ 0.  0.]
     [ 0.  0.]]]]



  [[[[ 9. 10.]
     [12. 13.]]

    [[ 0.  0.]
     [ 0.  0.]]]


   [[[ 0.  0.]
     [ 0.  0.]]

    [[ 0.  0.]
     [ 0.  0.]]]]



  [[[[18. 19.]
     [21. 22.]]

    [[ 0.  0.]
     [ 0.  0.]]]


   [[[ 0.  0.]
     [ 0.  0.]]

    [[ 0.  0.]
     [ 0.  0.]]]]]]
[[[[[[ 0.  1.]
     [ 3.  4.]]

    [[ 1.  2.]
     [ 4.  5.]]]


   [[[ 0.  0.]
     [ 0.  0.]]

    [[ 0.  0.]
     [ 0.  0.]]]]



  [[[[ 9. 10.]
     [12. 13.]]

    [[10. 11.]
     [13. 14.]]]


   [[[ 0.  0.]
     [ 0.  0.]]

    [[ 0.  0.]
     [ 0.  0.]]]]



  [[[[18. 19.]
     [21. 22.]]

    [[19. 20.]
     [22. 23.]]]


   [[[ 0.  0.]
     [ 0.  0.]]

    [[ 0.  0.]
     [ 0.  0.]]]]]]
[[[[[[ 0.  1.]
     [ 3.  4.]]

    [[ 1.  2.]
     [ 4.  5.]]]



## CNN층 구현하기

In [7]:
class Convolution:
    def __init__(self, W, b, stride=1, pad=0):
        self.W = W
        self.b = b
        self.stride = stride
        self.pad = pad
        
    def forward(self, x):
        FN, C, FH, FW = self.W.shape
        N, C, H, W = x.shape
        
        out_h = int((H + 2*self.pad - FH) / self.stride + 1)
        out_W = int((W + 2*self.pad - FW) / self.stride + 1)
        
        col = im2col(x, FH, FW, self.stride, self.pad)
        col_W = self.W.reshape(FN, -1).T
        out = np.dot(col, col_W) + self.b
        
        out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)
        
        return out
    
    def backward(self, dout):
        FN, C, FH, FW = self.W.shape
        dout = dout.transpose(0,2,3,1).reshape(-1, FN)
        
        self.db = np.sum(dout, axis=0)
        self.dW = np.dot(self.col.T, dout)
        self.dW = self.dW.transpose(1,0).reshape(FN, C, FH, FW)
        
        dcol = np.dot(dout, self.col_W.T)
        dx = col2im(dcol, self.x.shape, FH, FW, self.stride, self.pad)

        return dx

In [8]:
class Pooling:
    def __init__(self, pool_h, pool_w, stride=1, pad=0) -> None:
        self.pool_h = pool_h
        self.pool_w = pool_w
        self.stride = stride
        self.pad = pad
        
    def forward(self, x):
        N, C, H, W = x.shape
        out_h = int((H + 2*self.pad - FH) / self.stride + 1)
        out_W = int((W + 2*self.pad - FW) / self.stride + 1)
        
        col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
        col = col.reshape(-1, self.pool_h*self.pool_w)
        
        out = np.max(col, axis=1)
        
        out = out.reshape(N, out_h, out_w, C).transpose(0,3,1,2)
        
        return out
    
    def backward(self, dout):
        pass

## CNN 구현하기

In [9]:
class SimpleConvNet:
    def __init__(self, input_dim=(1, 28, 28), 
                 conv_params={'filter_num':30, 
                             'filter_size':5,
                             'pad':0,
                             'stride':1},
                 hidden_size=100, output_size=30, weight_init_std=0.01) -> None:
        
        filter_num = conv_params['filter_num']
        filter_size = conv_params['filter_size'] ## 정사각형 모양의 필터만 사용하는구만
        filter_pad = conv_params['pad']
        filter_stride = conv_params['stride']
        
        conv_output_size = (input_size - filter_size + 2*filter_pad) / filter_stride + 1
        pool_output_size = int(filter_num * (conv_output_size/2) * (conv_output_size/2))
        
        
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(filter_num, input_dim[0], filter_size, filter_size)
        self.params['b1'] = np.zeros(filter_num)
        self.params['W2'] = weight_init_std * np.random.randn(pool_output_size, hidden_size)
        self.params['b2'] = np.zeros(hidden_size)
        self.params['W3'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b3'] = np.zeros(output_size)
        
        
        self.layers = OrderedDict()
        self.layers['Conv1'] = Convolution(self.params['W1'], self.params['b1'], conv_params['stride'], conv_params['pad'])
        self.layers['Relu1'] = Relu()
        self.layers['Pool1'] = Pooling(pool_h=2, pool_w=2, stride=2)
        self.layers['Affine1'] = Affine(self.params['W2'], self.params['b2'])
        self.layers['Relu2'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W3'], self.params['b3'])
        
        self.last_layer = SoftmaxWithLoss()
        
        
        
        def predict(self, x):
            for layer in layers.values():
                x = layer.forward(x)
            return x
        
        def loss(self, x, t):
            y = self.predict(x)
            return self.last_layer.forward(y, t)
        
        def gradient(self, x, t):
            self.loss(x, t)
            
            dout = 1
            dout = self.last_layer.backward(dout)
            
            layers = list(self.layers.values())
            layers.reverse()
            for layer in layers:
                dout = layer.backward(dout)
                
            grads = {}
            grads['W1'] = self.layers['Conv1'].dW
            grads['b1'] = self.layers['Conv1'].db
            grads['W2'] = self.layers['Affine1'].dW
            grads['b2'] = self.layers['Affine1'].db
            grads['W3'] = self.layers['Affine2'].dW
            grads['b3'] = self.layers['Affine2'].db
            
            return grads
        
        